# Continue Batching

连续批处理(Continue Batching) 是一种 LLM 推理服务加速技术, 其主要解决批解码(Batching Decoding)中某些请求提前结束，导致批解码空闲问题。连续批处理是 vLLM 的关键组件技术。

在实现关键在于：

1. 主循环 continue batching inference 固定 prefill/decoding 操作
2. 求类管理 prompt 状态, 主循环中监听请求, 若有新请求则进行 prefill, 若有正在处理的 prompt 则进行 decoding
4. 增加 KV-Cache 管理：a. 初始化 batching_size, seq_len b. 状态表(使用、空闲) c. 索引(requestid, batch_id) d. 批内最大长度

伪代码为:

```python
requestor_num = 1000
reqs = Requestor(N = requestor_num)

# 与显存相关
KVengines = KVCacheEngine(batch_size, 
                          max_seq_len,)
model = model(KVengines)
Inferencer = ContinueBatchingInference(model, reqs)
next_token=torch.zeros(batch_size, 1, dtype=torch.long)

# Inferencer::Step() 主循环
while(not reqs.is_no_empty()):
  if model.kvengines.is_process():
    # do decoding
    decoding_logits = model.forward_continue_batching_decoding(next_token)
    
  if model.kvengines.is_avalable():
    prompts = reqs.get_prompts()
    if len(prompts) != 0:
      # do prefill
      prefill_logist = model.forward_conitnue_batching_prefill(prompts)
      
  # predict next token
  next_tokens = self.generate(decoding_logis, prefill_logits)
      
  # update kv-cache engines
  model.kvengines.update()
  next_tokens, req_ids = model.kvengines.rearrange(next_tokens)
  
  # reqs output
  reqs.update(req_ids, next_tokens)
```

代码实际实现逻辑与上述伪代码有一定出入

In [388]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Dict, List, Set, Tuple

torch.manual_seed(42)

## config

In [389]:
from dataclasses import dataclass

# @dataclass
# class ContinueBatchingEngineConfig:
#     request_size = 1000
#     max_batch_size = 16
#     max_seq_len = 512
#     max_prompt_len:int = 128
    
#     # model & kv cache
#     num_layers = 3
    
#     dim = 128
#     num_heads = 2
#     head_dim = 64
#     vocab_size = 100

# config = ContinueBatchingEngineConfig()
# print(config.max_seq_len)


@dataclass
class ContinueBatchingEngineConfig:
    request_size = 1000
    max_batch_size = 4
    max_seq_len = 32
    max_prompt_len:int = 16
    
    # model & kv cache
    num_layers = 3
    
    dim = 16
    num_heads = 2
    head_dim = 8
    vocab_size = 20

config = ContinueBatchingEngineConfig()
print(config.max_seq_len)

32


## Request

In [414]:
EOS_TOKEN = 0

In [415]:
class Request:
    def __init__(self, 
                 request_id:int, 
                 prompt: List[int], 
                 max_len: int = 2048):
        self.request_id = request_id
        self.prompt = prompt
        self.generated_tokens = []
        self.status = "REQUEST_WAITING"  # waiting, running, completed
        self.current_length = len(prompt)
        self.max_length = max_len
        
    def add_token(self, token: int):
        """添加生成的token到请求中"""
        self.generated_tokens.append(token)
        self.current_length += 1
        if self.is_finished():
            self.status = "REQUEST_COMPLETED"
            print(f'finished: request ID.{self.request_id}, generate new-tokens len:{len(self.generated_tokens)}')
    
    def is_finished(self) -> bool:
        """检查请求是否完成（达到最大长度或生成了EOS）"""
        return (self.current_length >= self.max_length or 
                (self.generated_tokens and self.generated_tokens[-1] == EOS_TOKEN))
    
    def get_full_sequence(self) -> List[int]:
        """获取完整的序列（prompt + 生成的tokens）"""
        return self.prompt + self.generated_tokens

req=Request(request_id=1, prompt=[1,2,3], max_len=10)
req.add_token(8)
req.get_full_sequence()

[1, 2, 3, 8]

## RequestManager

In [416]:
from queue import deque

class RequestManager:
    """管理所有请求的调度和状态"""
    
    def __init__(self, max_batch_size: int):
        self.max_batch_size = max_batch_size
        self.requests = {}  # request_id -> Request
        self.waiting_queue = deque()
        self.running_requests = set()
        self.next_request_id = 0
        
    def add_request(self, prompt: List[int], max_seq_len: int) -> int:
        """添加新请求，返回请求ID"""
        request_id = self.next_request_id
        self.next_request_id += 1
        request = Request(request_id, prompt, max_seq_len)
        self.requests[request_id] = request
        self.waiting_queue.append(request_id)
        return request_id
    
    def get_available_slots(self) -> int:
        """获取可用的批次空位数量"""
        return self.max_batch_size - len(self.running_requests)
    
    def get_pending_requests(self, max_count: int) -> List[Tuple[int, List[int]]]:
        """获取等待处理的请求"""
        available_slots = self.get_available_slots()
        count = min(max_count, available_slots, len(self.waiting_queue))
        
        requests_to_process = []
        for _ in range(count):
            if not self.waiting_queue:
                break
            request_id = self.waiting_queue.popleft()
            request = self.requests[request_id]
            request.status = "REQUEST_RUNNING"
            self.running_requests.add(request_id)
            requests_to_process.append((request_id, request.prompt))
            
        return requests_to_process
    
    def update_request(self, request_id: int, next_token: int):
        """更新请求状态"""
        if request_id in self.requests:
            request = self.requests[request_id]
            request.add_token(next_token)
            if request.is_finished():
                self.running_requests.discard(request_id)
    
    def has_pending_requests(self) -> bool:
        """检查是否有未完成的请求"""
        return len(self.waiting_queue) > 0 or len(self.running_requests) > 0
        
    def get_num_pending_requests(self) -> bool:
        return len(self.waiting_queue)
    
    def get_running_request_ids(self) -> List[int]:
        """获取当前正在运行的请求ID"""
        return list(self.running_requests)

requestor=RequestManager(max_batch_size=config.max_batch_size)
print(requestor.get_available_slots())
print(requestor.add_request(prompt=[1,2,3], max_seq_len=config.max_seq_len))
print(requestor.has_pending_requests())
print(requestor.get_running_request_ids())

4
0
True
[]


## KVCacheManager

In [417]:
class KVCacheManager:
    """管理Transformer的KV缓存"""
    
    def __init__(self, config):
        # 初始化KV缓存 [layer, batch, seq, head, dim]
        
        self.k_cache = torch.zeros(config.num_layers, config.max_batch_size, 
                                   config.max_seq_len, config.num_heads, config.head_dim)
        self.v_cache = torch.zeros(config.num_layers, config.max_batch_size, 
                                   config.max_seq_len, config.num_heads, config.head_dim)
        
        self.sequence_lengths = torch.zeros(config.max_batch_size, dtype=torch.long)
        self.request_to_slot = {}  # request_id -> slot_index
        self.slot_to_request = {}  # slot_index -> request_id
        self.free_slots = set(range(config.max_batch_size))
        
    def has_active_requests(self) -> bool:
        """检查是否有活跃的请求"""
        return len(self.slot_to_request) > 0
    
    def has_available_slots(self) -> bool:
        """检查是否有可用的槽位"""
        return len(self.free_slots) > 0
        
    def get_available_slots(self) -> bool:
        """检查是否有可用的槽位"""
        return len(self.free_slots)
    
    def allocate_slots(self, request_ids: List[int]) -> List[int]:
        """为请求分配槽位"""
        allocated_slots = []
        for request_id in request_ids:
            if not self.free_slots:
                break
            slot_id = self.free_slots.pop()
            self.request_to_slot[request_id] = slot_id
            self.slot_to_request[slot_id] = request_id
            self.sequence_lengths[slot_id] = 0
            allocated_slots.append(slot_id)
        return allocated_slots
    
    def free_slot(self, request_id: int):
        """释放请求占用的槽位"""
        if request_id in self.request_to_slot:
            slot_id = self.request_to_slot[request_id]
            del self.request_to_slot[request_id]
            del self.slot_to_request[slot_id]
            self.free_slots.add(slot_id)
            # 清空该槽位的缓存
            self.k_cache[:, slot_id, :, :, :] = 0
            self.v_cache[:, slot_id, :, :, :] = 0

    def update_slots(self, slot_ids, new_kv_cache):
        # print('update_slots', slot_ids)
        for i, layer_kv_cache in enumerate(new_kv_cache):
            bsz, seq_len, num_heads, head_dim = layer_kv_cache[0].shape
            if seq_len == 1: # decoding mode
                cur_len = self.sequence_lengths[slot_ids]
                self.k_cache[i, slot_ids, cur_len, :, :]  = layer_kv_cache[0][:,0,:,:]
                self.v_cache[i, slot_ids, cur_len, :, :]  = layer_kv_cache[1][:,0,:,:]
            else: # Prefill mode
                self.k_cache[i, slot_ids, :seq_len, :, :]  = layer_kv_cache[0]
                self.v_cache[i, slot_ids, :seq_len, :, :]  = layer_kv_cache[1]
                
    
    def update_after_step(self, new_tokens: torch.Tensor):
        """更新步进后的序列长度"""
        active_slots = list(self.slot_to_request.keys())
        self.sequence_lengths[active_slots] += 1
    
    def get_active_slots_info(self) -> Tuple[List[int], List[int]]:
        """获取活跃槽位的信息"""
        active_slots = list(self.slot_to_request.keys())
        request_ids = [self.slot_to_request[slot] for slot in active_slots]
        return active_slots, request_ids
    
    def get_kv_cache_for_slots(self, slot_ids: List[int]) -> Tuple[torch.Tensor, torch.Tensor]:
        """获取指定槽位的KV缓存"""
        return self.k_cache[:, slot_ids], self.v_cache[:, slot_ids]

cacher = KVCacheManager(config)
print(cacher.k_cache.shape)
print(cacher.has_available_slots())

cacher.allocate_slots([18, 19, 13])
print(cacher.get_active_slots_info())

cacher.free_slot(1)
print(cacher.get_active_slots_info())

cacher.free_slot(13)
print(cacher.get_active_slots_info())

torch.Size([3, 4, 32, 2, 8])
True
([0, 1, 2], [18, 19, 13])
([0, 1, 2], [18, 19, 13])
([0, 1], [18, 19])


## Model

In [418]:
import math
class DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_heads
        self.dim = config.dim
        self.head_dim = config.head_dim
        self.WQ = nn.Linear(config.dim, config.dim, bias=False)
        self.WK = nn.Linear(config.dim, config.dim, bias=False)
        self.WV = nn.Linear(config.dim, config.dim, bias=False)
        self.WO = nn.Linear(config.dim, config.dim, bias=False)
        self.act = nn.ReLU()
        
    def forward(self, X, kvcache=None, current_length=None):
        bsz, seq_len, _ = X.shape
        Q, K, V= self.WQ(X), self.WK(X), self.WV(X)
        Q=Q.reshape(bsz, seq_len, self.num_heads, self.head_dim).transpose(1,2)
        K=K.reshape(bsz, seq_len, self.num_heads, self.head_dim)
        V=V.reshape(bsz, seq_len, self.num_heads, self.head_dim)

        if kvcache is None:
            K_, V_ = K, V
        else:
            # print('kv cache cat:', kvcache[0].shape, K.shape)
            K_ = torch.cat((kvcache[0], K), dim = 1)
            V_ = torch.cat((kvcache[1], V), dim = 1)


        K_ = K_.transpose(1,2)
        V_ = V_.transpose(1,2)

        S = Q@K_.transpose(2,3)//math.sqrt(self.head_dim)
        P = F.softmax(S, dim = -1)
        Z = P@V_
        Z = Z.transpose(1,2).reshape(bsz, seq_len, self.dim)
        O = self.WO(Z)

        # activate & shorcut
        O_ = X + self.act(O)
        
        return O_, [K, V]

In [419]:
class ToyModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embd = nn.Embedding(config.vocab_size, config.dim)
        self.lm_head = nn.Linear(config.dim, config.vocab_size)
        self.decoder = nn.ModuleList(
            [ DecoderBlock(config) for i in range(config.num_layers)]
        )

    def forward(self, x, kvcaches=None, current_length=None):
        layer_kvcaches=[]
        X = self.embd(x)
        
        for i, block in enumerate(self.decoder):
            if kvcaches == None:
                X, layer_kvcache = block(X, None, None)
            else:
                X, layer_kvcache = block(X, kvcache=[kvcaches[0][i], kvcaches[1][i]], current_length=current_length)
                
            layer_kvcaches.append(layer_kvcache)
        logits = self.lm_head(X)
        return logits, layer_kvcaches

In [420]:
model=ToyModel(config)
data = torch.randint(config.vocab_size, (2,3))
print(model)
print(model(data)[0].shape)
print(len(model(data)[1]))

ToyModel(
  (embd): Embedding(20, 16)
  (lm_head): Linear(in_features=16, out_features=20, bias=True)
  (decoder): ModuleList(
    (0-2): 3 x DecoderBlock(
      (WQ): Linear(in_features=16, out_features=16, bias=False)
      (WK): Linear(in_features=16, out_features=16, bias=False)
      (WV): Linear(in_features=16, out_features=16, bias=False)
      (WO): Linear(in_features=16, out_features=16, bias=False)
      (act): ReLU()
    )
  )
)
torch.Size([2, 3, 20])
3


## ModelWrapper

In [421]:
class ModelWrapper:
    """封装模型的前向传播"""
    
    def __init__(self, model, kv_cache_manager: KVCacheManager):
        self.model = model
        self.kv_cache_manager = kv_cache_manager
    
    def prefill_requests(self, requests: List[Tuple[int, List[int]]]) -> torch.Tensor:
        """预填充新请求"""
        if not requests:
            return torch.tensor([])
            
        # 分配槽位
        request_ids = [req_id for req_id, _ in requests]
        slot_ids = self.kv_cache_manager.allocate_slots(request_ids)
        
        # 准备输入 - 将不同长度的 prompt 填充到相同长度
        prompts = [prompt for _, prompt in requests]
        max_len = max(len(prompt) for prompt in prompts)
        
        input_ids = torch.zeros(len(prompts), max_len, dtype=torch.long)
        # attention_mask = torch.zeros(len(prompts), max_len, dtype=torch.long)
        
        for i, prompt in enumerate(prompts):
            # decoding 通常要做 left padding
            input_ids[i, :len(prompt)] = torch.tensor(prompt)
        
        # 执行预填充
        with torch.no_grad():
            logits, layer_kvcaches = self.model(input_ids,)
        
        # 更新KV缓存
        self._update_kv_cache(slot_ids, layer_kvcaches)
        
        # 返回最后一个token的logits
        return logits[:, -1, :].unsqueeze(1) # bsz, seq_len, vocab_size
    
    def decode_next_tokens(self, next_tokens: torch.Tensor, slot_ids: List[int], current_length) -> torch.Tensor:
        """解码下一个token"""
        if len(slot_ids) == 0:
            return torch.tensor([])

        with torch.no_grad():
            logits, layer_kvcaches = self.model(
                next_tokens,
                kvcaches = self.kv_cache_manager.get_kv_cache_for_slots(slot_ids),
                current_length = current_length,
            )
        
        # 更新KV缓存
        self._update_kv_cache(slot_ids, layer_kvcaches)
        
        return logits
    
    def _update_kv_cache(self, slot_ids: List[int], new_kv_cache):
        """更新KV缓存"""
        self.kv_cache_manager.update_slots(slot_ids, new_kv_cache)
        return
        
    def generate_next_tokens(self, logits: torch.Tensor) -> torch.Tensor:
        """从logits生成下一个token（贪婪采样）"""
        if len(logits) == 0:
            return torch.tensor([])
        return torch.argmax(logits, dim=-1)

## ContinueBatchingEngine

In [422]:
class ContinueBatchingEngine:
    """连续批处理主引擎"""
    
    def __init__(self, model, config):
        self.kv_cache_manager = KVCacheManager(config)
        self.model_wrapper = ModelWrapper(model, self.kv_cache_manager)
        self.request_manager = RequestManager(config.max_batch_size,)
    
    def add_request(self, prompt: List[int], max_seq_len) -> int:
        """添加新请求"""
        return self.request_manager.add_request(prompt, max_seq_len)
    
    def step(self):
        # 阶段2: 处理解码（已有请求）
        if self.kv_cache_manager.has_active_requests():
            active_slots, request_ids = self.kv_cache_manager.get_active_slots_info()

            # 准备输入token (上一个步骤生成的token)
            input_tokens = torch.tensor([
                self.request_manager.requests[req_id].generated_tokens[-1]
                for req_id in request_ids
            ], dtype=torch.long)

            current_length = torch.tensor([
                self.request_manager.requests[req_id].current_length
                for req_id in request_ids
            ], dtype=torch.long)

            input_tokens = input_tokens.unsqueeze(dim = 1)
            
            # 解码
            decoding_logits = self.model_wrapper.decode_next_tokens(input_tokens, active_slots, current_length)
            next_tokens = self.model_wrapper.generate_next_tokens(decoding_logits)
            
            # 更新状态
            self.kv_cache_manager.update_after_step(next_tokens)
            for i, request_id in enumerate(request_ids):
                self.request_manager.update_request(request_id, next_tokens[i].item())
                
                # 如果请求完成，释放槽位
                if self.request_manager.requests[request_id].is_finished():
                    self.kv_cache_manager.free_slot(request_id)
                    
        # 阶段1: 处理预填充（新请求）
        if self.kv_cache_manager.has_available_slots():
            pending_requests = self.request_manager.get_pending_requests(
                self.kv_cache_manager.get_available_slots()
            )
            if pending_requests:
                prefill_logits = self.model_wrapper.prefill_requests(pending_requests)
                prefill_tokens = self.model_wrapper.generate_next_tokens(prefill_logits)
                
                # 更新请求状态
                for i, (request_id, _) in enumerate(pending_requests):
                    self.request_manager.update_request(request_id, prefill_tokens[i].item())
                    
    def has_pending_work(self) -> bool:
        """检查是否还有未完成的工作"""
        return self.request_manager.has_pending_requests()

    def get_requests_info(self):
        pending=self.request_manager.get_num_pending_requests()
        total_request=len(self.request_manager.requests)
        return pending, total_request

## Run

In [423]:
from random import randint
def listen_request(config, p=0.01):
    prompt=[]
    prompt_len=0
    num = randint(1,100)
    if num/100.0 < p:
        prompt_len = randint(config.max_prompt_len//4, config.max_prompt_len)
        prompt=torch.randint(config.vocab_size, (1, prompt_len))
        prompt=prompt[0].tolist()
    return prompt, prompt_len

In [424]:
model = ToyModel(config)
worker = ContinueBatchingEngine(model, config)

In [425]:
N = 100
count = 0

# main 
while 1:
    # 监听进程
    if count != N:
        prompt, prompt_len = listen_request(config, p=0.5)
        if prompt_len != 0:
            count += 1
            if count % (N//10) == 0:
                per = count / (N//10) 
                print('Running...:','*'*int(per),'-'*(10-int(per)))
            generate_len = randint(prompt_len, config.max_seq_len)
            worker.add_request(prompt, generate_len)
            pending, total = worker.get_requests_info()
            print(f'Request Info: pending:{pending}, total:{total}, N:{N}')
            
    # 处理进程   
    worker.step()

    if not worker.has_pending_work() and count == N:
        print('process done')
        break

Request Info: pending:1, total:1, N:100
Request Info: pending:1, total:2, N:100
Request Info: pending:1, total:3, N:100
finished: request ID.2, generate new-tokens len:2
Request Info: pending:1, total:4, N:100
finished: request ID.3, generate new-tokens len:2
Request Info: pending:1, total:5, N:100
Request Info: pending:1, total:6, N:100
Request Info: pending:1, total:7, N:100
Request Info: pending:2, total:8, N:100
finished: request ID.0, generate new-tokens len:13
Request Info: pending:2, total:9, N:100
Running...: * ---------
Request Info: pending:3, total:10, N:100
Request Info: pending:4, total:11, N:100
finished: request ID.4, generate new-tokens len:12
finished: request ID.7, generate new-tokens len:1
finished: request ID.7, generate new-tokens len:2
Request Info: pending:3, total:12, N:100
finished: request ID.6, generate new-tokens len:10
Request Info: pending:3, total:13, N:100
Request Info: pending:4, total:14, N:100
finished: request ID.1, generate new-tokens len:23
Request